# GPTWake — Chuẩn bị dataset ngay trên Kaggle

Notebook này **tải dữ liệu của nhóm từ GitHub một lần** rồi tạo sẵn dataset cho hai notebook train
(`m5_wake_word.ipynb`, `sincconv_wake_word.ipynb`). Mọi thứ chạy trên máy Kaggle, không cần ổ cứng hay mạng của bạn.

### Vì sao làm trên Kaggle?

- **Ổ cứng:** tải đủ dữ liệu về máy cần khoảng 13 GB (file WAV + bản cache của Git LFS). Máy Kaggle có ổ đủ lớn.
- **Băng thông Git LFS có hạn:** GitHub tính băng thông tải file LFS vào tài khoản **chủ repo**, gói miễn phí khoảng
  10 GiB mỗi tháng; vượt thì LFS có thể bị chặn tới đầu tháng sau. Một lần tải đủ dữ liệu là khoảng 6 GB.
  Vì vậy **cả nhóm chỉ nên tải một lần**, ở đây, rồi dùng chung dataset Kaggle. Đừng ai `git lfs pull` toàn bộ nữa.
- **Không phải upload:** dữ liệu tải thẳng về máy Kaggle, không đi qua mạng nhà bạn.

### Cách chạy

1. Kaggle → *Create → New Notebook* → *File → Import Notebook* → chọn file này.
2. Bảng bên phải: *Settings → Internet* **bật** (tài khoản Kaggle cần xác minh số điện thoại mới bật được).
   Accelerator để *None* (không cần GPU, đỡ tốn hạn mức GPU).
3. *Save Version → Save & Run All (Commit)*. Chạy nền khoảng 15–40 phút tuỳ mạng.
4. Chạy xong, mở trang notebook → tab *Output*: thấy thư mục `gptwake/` gồm `validation/`, `holdout/`,
   `train/H00.csv` … `train/H70.csv`, `train_pool/`.
5. Dùng output này cho notebook train, một trong hai cách:
   - Trong notebook train: *Add Input* → chọn **Notebook Output** → tìm notebook này. Notebook train tự tìm dữ liệu.
   - Hoặc ở tab *Output* của notebook này, tạo **New Dataset** từ output, rồi đặt dataset ở chế độ Public.
6. Muốn cả nhóm dùng: đặt notebook (hoặc dataset) ở chế độ **Public**, hoặc thêm các thành viên làm collaborator.

## 1. Cấu hình

- `REPO_URL`, `BRANCH`: repo và nhánh lấy dữ liệu (repo của nhóm đang public).
- `LFS_INCLUDE`: chỉ tải các thư mục mà script tạo train/validation/holdout cần (khoảng 5,8 GB). Không tải
  `negative/raw` (3,7 GB bản ghi dài) và các thư mục khác.
- `SPLITS`: các phân phối train cần tạo, theo số của `scripts/create_train.py` (1 = H00, 2 = H25, 3 = H50, 4 = H70).
- `WORK_DIR`: nơi clone repo (thư mục tạm, không nằm trong output). `OUT_DIR`: nơi ghi dataset (nằm trong output).
- `DOWNLOAD`, `MAKE_SPLITS`: đặt `False` để bỏ qua bước tương ứng khi chạy lại.

In [ ]:
REPO_URL = "https://github.com/hqcfmlno1/GPTWake.git"
BRANCH = "main"
LFS_INCLUDE = [
    "data/positive/augmented/**",
    "data/hard_negative/augmented/**",
    "data/negative/processed/**",
    "data/negative/augmented/**",
    "data/validation/**",
    "data/holdout/**",
]
SPLITS = ["1", "2", "3", "4"]
WORK_DIR = "/tmp/GPTWake"
OUT_DIR = "/kaggle/working/gptwake"
DOWNLOAD = True
MAKE_SPLITS = True

## 2. Kiểm tra môi trường

Cần `git`, `git-lfs` và đủ chỗ trống. Nếu máy chưa có `git-lfs`, ô dưới cài bằng `apt-get` (cần Internet).
Dung lượng: lúc tải cần khoảng 12 GB ở thư mục tạm (file + cache LFS; cache được xoá ngay sau đó), output khoảng 5 GB.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None, env=None):
    """Chạy lệnh shell, in output ra ngay; dừng notebook nếu lệnh lỗi."""
    print("$", cmd)
    result = subprocess.run(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})})
    if result.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {result.returncode}): {cmd}")


def free_gb(path):
    path = Path(path)
    while not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free / 1e9


if shutil.which("git-lfs") is None:
    run("apt-get -qq update && apt-get -qq install -y git-lfs")
run("git --version && git lfs version")
print(f"Trống ở thư mục tạm: {free_gb(WORK_DIR):.1f} GB; ở output: {free_gb(OUT_DIR):.1f} GB")

## 3. Clone repo và tải đúng phần dữ liệu cần

1. Clone với `GIT_LFS_SKIP_SMUDGE=1`: chỉ lấy code và các file "con trỏ" LFS (vài trăm byte mỗi file), chưa tải WAV.
2. `git lfs pull --include …`: tải nội dung WAV thật, chỉ cho các thư mục trong `LFS_INCLUDE`.
3. Xoá cache `.git/lfs/objects` (một bản sao thứ hai của mọi WAV) để lấy lại khoảng 6 GB.
4. Đếm lại: mọi file WAV trong các thư mục cần phải là WAV thật, không còn file con trỏ.

In [ ]:
work = Path(WORK_DIR)
if DOWNLOAD:
    if work.exists():
        shutil.rmtree(work)
    run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {work}", env={"GIT_LFS_SKIP_SMUDGE": "1"})
    run("git lfs install --local", cwd=work)
    run(f'git lfs pull --include="{",".join(LFS_INCLUDE)}"', cwd=work)
    shutil.rmtree(work / ".git" / "lfs" / "objects", ignore_errors=True)


def is_real_wav(path):
    with open(path, "rb") as f:
        head = f.read(12)
    return head[:4] in (b"RIFF", b"RF64") and head[8:12] == b"WAVE"


for pattern in LFS_INCLUDE:
    folder = work / pattern.removesuffix("/**")
    files = list(folder.rglob("*.wav"))
    real = sum(is_real_wav(p) for p in files)
    print(f"{folder.relative_to(work)}: {real:,}/{len(files):,} WAV thật")
    if real != len(files):
        raise RuntimeError(f"{folder} còn file con trỏ LFS: tải chưa xong (hết hạn mức LFS?). Xem log ở trên.")
print(f"Còn trống ở thư mục tạm: {free_gb(WORK_DIR):.1f} GB")

## 4. Tạo bốn phân phối train

Chạy `scripts/create_train.py` của repo với từng số trong `SPLITS`. Script tạo `data/train/Hxx/` bằng symlink tới file gốc
và tự kiểm tra validation/holdout khớp lựa chọn cố định của nhóm (seed 42).

In [ ]:
if MAKE_SPLITS:
    for choice in SPLITS:
        run(f"{sys.executable} scripts/create_train.py {choice}", cwd=work)
print(sorted(p.name for p in (work / "data" / "train").iterdir()))

## 5. Xuất dataset

Ô đầu là **nguyên văn** `scripts/export_kaggle_dataset.py` của repo (ghi ra file tạm), ô sau chạy nó với `--copy`:
- chép `validation/`, `holdout/`;
- chép mỗi file train đúng một lần vào `train_pool/positive|negative/`;
- ghi danh sách file của từng phân phối vào `train/Hxx.csv`.

Kết quả nằm trong `OUT_DIR` (thuộc output của notebook), không còn symlink.

In [ ]:
%%writefile /tmp/export_kaggle_dataset.py
"""Gom validation, holdout và các phân phối train thành một thư mục sẵn sàng upload lên Kaggle.

``scripts/create_train.py`` tạo ``data/train/Hxx`` bằng symlink. Kaggle không upload symlink,
và bốn phân phối H00-H70 dùng chung phần lớn file. Script này:

- chép ``data/validation`` và ``data/holdout`` sang thư mục đích;
- chép mỗi file train đúng một lần vào ``train_pool/{positive,negative}/``;
- ghi danh sách file của từng phân phối vào ``train/Hxx.csv`` (cột ``path``, ``label``,
  đường dẫn tính từ thư mục ``train/``).

Notebook ``notebooks/m5_wake_word.ipynb`` và ``notebooks/sincconv_wake_word.ipynb`` đọc
được cấu trúc này. Mặc định dùng hard link (không tốn thêm dung lượng, cần cùng ổ đĩa);
nếu không được thì sao chép.

Ví dụ:
    python scripts/create_train.py 1     # tạo data/train/H00 (làm với 2, 3, 4 tuỳ nhu cầu)
    python scripts/export_kaggle_dataset.py   # ghi ra data/kaggle_dataset/ (Git bỏ qua data/**)
"""

import argparse
import csv
import json
import os
import shutil
from pathlib import Path

LABELS = {"positive": 1, "negative": 0}


def check_wav(path: Path):
    with path.open("rb") as stream:
        header = stream.read(12)
    if header[:4] not in (b"RIFF", b"RF64") or header[8:12] != b"WAVE":
        raise ValueError(f"{path} không phải WAV. Đã chạy `git lfs pull` chưa?")


def place(source: Path, target: Path, copy: bool) -> str:
    """Đặt file `source` (đã giải symlink) tại `target`. Trả về cách đã dùng."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if os.path.samefile(source, target) or target.read_bytes() == source.read_bytes():
            return "có sẵn"
        raise FileExistsError(f"{target} đã có và khác {source}.")
    if not copy:
        try:
            os.link(source, target)
            return "hard link"
        except OSError:
            pass
    shutil.copy2(source, target)
    return "sao chép"


def export_split(source_dir: Path, target_dir: Path, copy: bool) -> int:
    count = 0
    for label in LABELS:
        for path in sorted((source_dir / label).glob("*.wav")):
            real = Path(os.path.realpath(path))
            check_wav(real)
            place(real, target_dir / label / path.name, copy)
            count += 1
    if not count:
        raise FileNotFoundError(f"Không có file WAV trong {source_dir}.")
    return count


def export_train(train_dir: Path, out: Path, copy: bool) -> int:
    rows = []
    for label, value in LABELS.items():
        for path in sorted((train_dir / label).glob("*.wav")):
            real = Path(os.path.realpath(path))
            check_wav(real)
            place(real, out / "train_pool" / label / path.name, copy)
            rows.append((f"../train_pool/{label}/{path.name}", value))
    if not rows:
        raise FileNotFoundError(f"Không có file WAV trong {train_dir}.")
    (out / "train").mkdir(parents=True, exist_ok=True)
    with (out / "train" / f"{train_dir.name}.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["path", "label"])
        writer.writerows(rows)
    return len(rows)


def main():
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", type=Path, default=root / "data")
    parser.add_argument("--out", type=Path, default=root / "data" / "kaggle_dataset")
    parser.add_argument(
        "--splits", nargs="*", default=None,
        help="Phân phối train cần xuất, ví dụ H00 H25. Mặc định: mọi thư mục trong data/train.",
    )
    parser.add_argument("--copy", action="store_true", help="Luôn sao chép thay vì hard link.")
    parser.add_argument("--title", default="GPTWake Hey GPT wake-word dataset")
    parser.add_argument("--id", default="YOUR_KAGGLE_USERNAME/gptwake", help="Kaggle dataset id")
    args = parser.parse_args()

    data, out = args.data_dir.resolve(), args.out.resolve()
    train_root = data / "train"
    splits = args.splits
    if splits is None:
        splits = sorted(p.name for p in train_root.iterdir() if p.is_dir()) if train_root.is_dir() else []
    if not splits:
        parser.exit(1, "Chưa có data/train/Hxx. Chạy `python scripts/create_train.py` trước.\n")

    for name in ("validation", "holdout"):
        print(f"{name}: {export_split(data / name, out / name, args.copy):,} file")
    for split in splits:
        print(f"train {split}: {export_train(train_root / split, out, args.copy):,} file")
    pool = sum(1 for _ in (out / "train_pool").rglob("*.wav"))
    print(f"train_pool: {pool:,} file duy nhất")

    metadata = out / "dataset-metadata.json"
    if not metadata.exists():
        metadata.write_text(
            json.dumps({"title": args.title, "id": args.id, "licenses": [{"name": "other"}]}, indent=2),
            encoding="utf-8",
        )
    print(f"Xong: {out}\nSửa 'id' trong {metadata.name} rồi upload (xem hướng dẫn trong notebook).")


if __name__ == "__main__":
    main()

In [ ]:
run(f"{sys.executable} /tmp/export_kaggle_dataset.py --data-dir {work / 'data'} --out {OUT_DIR} --copy")

out = Path(OUT_DIR)
total = sum(p.stat().st_size for p in out.rglob("*") if p.is_file())
print(f"\nDataset: {out} — {sum(1 for _ in out.rglob('*.wav')):,} file WAV, {total / 1e9:.2f} GB")
for path in sorted(out.iterdir()):
    print(" ", path.name)
for csv_path in sorted((out / "train").glob("*.csv")):
    print(f"  train/{csv_path.name}: {sum(1 for _ in open(csv_path)) - 1:,} dòng")

## 6. Bước tiếp theo

- *Save Version → Save & Run All* (nếu bạn đang chạy tương tác), đợi xong.
- Trong notebook train (M5 hoặc SincConv): *Add Input → Notebook Output* → chọn notebook này, bật GPU, chạy.
  Notebook train tự tìm `validation/`, `holdout/` và `train/Hxx.csv` trong `/kaggle/input`.
- Chia sẻ cho nhóm: đặt notebook này (hoặc dataset tạo từ output của nó) ở chế độ Public.
- **Đừng chạy lại notebook này nhiều lần trong tháng**: mỗi lần chạy tải khoảng 6 GB từ băng thông LFS của chủ repo.